# Exploring sample trip logs & testing PySpark logic

In [1]:
import requests
import zipfile
import io
import os
from pyspark.sql import SparkSession
import glob
import xml.etree.ElementTree as ET

## Extracting more recent datasets

In [2]:
# Define local mock Azure Data Lake path
bronze_historical_path = "../data/bronze/historical/"
os.makedirs(bronze_historical_path, exist_ok=True)

# Define the sample months (Mix of older nested formats and modern formats)
zip_urls = [
    #"https://s3.amazonaws.com/tripdata/2014-citibike-tripdata.zip", # Older nested format
    "https://s3.amazonaws.com/tripdata/2022-citibike-tripdata.zip"
    #"https://s3.amazonaws.com/tripdata/202407-citibike-tripdata.zip"  # Modern format
]


### download script flattens folder embeddings

In [3]:
def extract_csvs_from_zip_urls(zip_urls):
    print("Downloading and extracting ZIP files...")
    for url in zip_urls:
        print(f"Fetching {url}...")
        response = requests.get(url)

        # Open the downloaded ZIP file in memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:

            # Iterate through every item inside the ZIP archive
            for file_path in z.namelist():

                file_name = os.path.basename(file_path)

                # Skip directories (which have no basename)
                if not file_name:
                    continue

                # Filtering Logic:
                # 1. Must be a .csv file
                # 2. Exclude Mac OS artifacts like __MACOSX or .DS_Store
                # 3. Exclude Jersey City data (files starting with 'JC-')
                if file_name.endswith('.csv') and '__MACOSX' not in file_path and not file_name.startswith('JC-'):

                    target_path = os.path.join(bronze_historical_path, file_name)
                    print(f"  -> Extracting {file_name}...")

                    # Read the specific CSV from the zip and write it directly to the target folder
                    # This flattens the structure, ignoring any internal ZIP folders
                    with z.open(file_path) as source, open(target_path, "wb") as target:
                        target.write(source.read())

    print(f"Extraction complete. Clean CSVs saved directly to {bronze_historical_path}")

In [4]:
extract_csvs_from_zip_urls(zip_urls)

Fetching https://s3.amazonaws.com/tripdata/2022-citibike-tripdata.zip...
Extraction complete. Clean CSVs saved directly to ../data/bronze/historical/


In [3]:
# Initialize local Spark session mimicking Databricks

spark = SparkSession.builder \
    .appName("CitiBike-Batch-EDA") \
    .getOrCreate()

In [4]:
# Read all extracted CSVs beginning with 2024 in the historical folder
# PySpark will automatically merge split CSVs from the same month
df_trips = spark.read.option("header", "true") \
    .csv(f"{bronze_historical_path}/2024*.csv")

# Profile the schema to map out data types
print("The new Schema:")
df_trips.printSchema()

The new Schema:
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)



In [5]:
df_trips.count()

4722896

In [6]:
df_trips.limit(20).toPandas()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,6C3563ED9BD36F2B,electric_bike,2024-07-10 09:12:44.192,2024-07-10 09:18:13.145,Front St & Jay St,4895.03,Dock 72 Way & Market St,4804.02,40.702461,-73.986842,40.69985,-73.97141,member
1,788C72113A42CACD,classic_bike,2024-07-12 07:35:39.714,2024-07-12 07:37:28.966,W 10 St & Washington St,5847.06,Perry St & Bleecker St,5922.07,40.73342399437081,-74.00851495563984,40.73535398,-74.00483091,member
2,239DEA356066DDF7,classic_bike,2024-07-04 12:59:46.344,2024-07-04 13:02:37.572,E 25 St & 2 Ave,6046.02,E 20 St & 2 Ave,5971.08,40.73912601,-73.97973776,40.73587678,-73.98205027,member
3,90B5EE27A7CCB271,electric_bike,2024-07-02 18:55:21.450,2024-07-02 18:59:12.023,E 3 St & 1 Ave,5553.03,Forsyth St & Grand St,5382.07,40.72467721,-73.98783413,40.71779817737835,-73.99316132068634,member
4,091DEE951A54DAF4,electric_bike,2024-07-14 05:25:11.691,2024-07-14 05:29:31.193,E 147 St & Bergen Ave,7840.11,E 135 St & St Ann's Ave,7687.05,40.814673,-73.91839,40.805089,-73.918889,casual
5,BEECD2A219E06D0C,electric_bike,2024-07-02 18:47:56.781,2024-07-02 19:11:45.810,Liberty St & Nassau St,5105.09,E 12 St & 3 Ave,5788.12,40.70858894,-74.00935482,40.73223272,-73.98889957,casual
6,C92090A948388A67,classic_bike,2024-07-07 21:42:04.190,2024-07-07 21:55:27.210,Cleveland Pl & Spring St,5492.05,Perry St & Bleecker St,5922.07,40.722103786686034,-73.99724900722504,40.73535398,-74.00483091,member
7,3EE42986A8D57E8B,electric_bike,2024-07-04 02:23:38.520,2024-07-04 03:15:32.053,W 17 St & 7 Ave,6107.08,Flatbush Ave & Ocean Ave,3704.04,40.74056423633952,-73.99852573871613,40.663657,-73.963014,member
8,6FDECEA3CABD1494,classic_bike,2024-07-06 15:06:46.151,2024-07-06 15:15:34.528,6 Ave & W 34 St,6364.10,FDR Drive & E 35 St,6230.04,40.74964,-73.98805,40.744219,-73.97121214,member
9,462DFEEDBBADBD0F,electric_bike,2024-07-06 22:11:38.818,2024-07-06 22:21:33.220,6 Ave & W 34 St,6364.10,W 27 St & 6 Ave,6215.07,40.74964,-73.98805,40.74526,-73.99062,casual


## Older data sets follow different folder embeddings and schemas

In [21]:
extract_csvs_from_zip_urls([zip_urls[0]])

Fetching https://s3.amazonaws.com/tripdata/2014-citibike-tripdata.zip...
  -> Extracting 201404-citibike-tripdata_1.csv...
  -> Extracting 201412-citibike-tripdata_1.csv...
  -> Extracting 201411-citibike-tripdata_1.csv...
  -> Extracting 201407-citibike-tripdata_1.csv...
  -> Extracting 201410-citibike-tripdata_1.csv...
  -> Extracting 201409-citibike-tripdata_1.csv...
  -> Extracting 201408-citibike-tripdata_1.csv...
  -> Extracting 201406-citibike-tripdata_1.csv...
  -> Extracting 201403-citibike-tripdata_1.csv...
  -> Extracting 201401-citibike-tripdata_1.csv...
  -> Extracting 201402-citibike-tripdata_1.csv...
  -> Extracting 201405-citibike-tripdata_1.csv...
Extraction complete. Clean CSVs saved directly to ../data/bronze/historical/


In [22]:
# Read all extracted CSVs beginning with 2014 in the historical folder
# PySpark will automatically merge split CSVs from the same month
df_trips_2014 = spark.read.option("header", "true") \
    .csv(f"{bronze_historical_path}/2014*.csv")

# Profile the schema to map out data types
print("The Older Schema:")
df_trips_2014.printSchema()

The Older Schema:
root
 |-- tripduration: string (nullable = true)
 |-- starttime: string (nullable = true)
 |-- stoptime: string (nullable = true)
 |-- start station id: string (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: string (nullable = true)
 |-- start station longitude: string (nullable = true)
 |-- end station id: string (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: string (nullable = true)
 |-- end station longitude: string (nullable = true)
 |-- bikeid: string (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: string (nullable = true)
 |-- gender: string (nullable = true)



### script to detect all different schemas present in the csv files

In [5]:
# Get all CSV files in the bronze directory
csv_files = glob.glob(f"{bronze_historical_path}/*.csv")

# Dictionary to group files by their exact column structure
schema_versions = {}

for file in csv_files:
    # PySpark reads ONLY the first row to get the header names
    # No full data scan is triggered here
    columns = tuple(spark.read.option("header", "true").csv(file).columns)

    file_name = os.path.basename(file)

    # Group the files by their schema tuple
    if columns not in schema_versions:
        schema_versions[columns] = []

    schema_versions[columns].append(file_name)

# Output the findings
print(f"Found {len(schema_versions)} different schema structures across the files:\n")

for i, (schema, files) in enumerate(schema_versions.items(), 1):
    print(f"=== Schema Version {i} ===")
    print(f"Total Columns: {len(schema)}")
    print(f"Columns: {list(schema)}")
    print(f"Files using this schema: {len(files)}")
    print(f"Sample Files: {files[:2]}\n")

NameError: name 'spark' is not defined

## all zip files url extraction script

In [24]:
def get_all_citibike_zip_urls():
    base_url = "https://s3.amazonaws.com/tripdata"
    zip_urls = []

    # S3 API uses a specific XML namespace we must reference
    ns = {'s3': 'http://s3.amazonaws.com/doc/2006-03-01/'}
    marker = None

    print("Fetching ZIP URLs from S3 bucket (excluding Jersey City)...")

    while True:
        params = {'marker': marker} if marker else {}
        response = requests.get(base_url, params=params)
        response.raise_for_status()

        root = ET.fromstring(response.content)

        for content in root.findall('s3:Contents', ns):
            key = content.find('s3:Key', ns).text

            # MODIFICATION: Must be a zip file AND must NOT start with 'JC'
            if key and key.endswith('.zip') and not key.startswith('JC'):
                zip_urls.append(f"{base_url}/{key}")

        is_truncated = root.find('s3:IsTruncated', ns)
        if is_truncated is not None and is_truncated.text.lower() == 'true':
            marker = root.findall('s3:Contents', ns)[-1].find('s3:Key', ns).text
        else:
            break

    print(f"Discovered {len(zip_urls)} non-JC ZIP files.")
    return zip_urls

In [25]:
all_urls = get_all_citibike_zip_urls()

Fetching ZIP URLs from S3 bucket (excluding Jersey City)...
Discovered 40 non-JC ZIP files.


In [26]:
print(all_urls)

['https://s3.amazonaws.com/tripdata/2013-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2014-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2015-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2016-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2017-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2018-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2019-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2020-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2021-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2022-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/2023-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/202402-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/202403-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripdata/202404-citibike-tripdata.zip', 'https://s3.amazonaws.com/tripd

 ### Combining the 3 scripts allows to fetch the url of all available citibike data and saving the csv files flat onto the bronze layer in the data lake and finally checking for schema changes before starting transformation
